In [0]:
%sql
USE CATALOG shopsphere;

CREATE TABLE IF NOT EXISTS shopsphere.silver.products(
    product_id INT,
   product_name STRING,
   category STRING,
   subcategory STRING,
   brand STRING,
   price DECIMAL,
   cost DECIMAL,
   supplier_id STRING,
   product_status STRING,
   updated_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS shopsphere.quarantine.products(
    product_id INT,
   product_name STRING,
   category STRING,
   subcategory STRING,
   brand STRING,
   price DECIMAL,
   cost DECIMAL,
   supplier_id STRING,
   product_status STRING,
   updated_at TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

df = spark.read.table("shopsphere.bronze.products")


In [0]:
#remove duplicates
df = df.dropDuplicates(["product_id"])

#Standardize product name
standard_name = df.withColumn("product_name", initcap(trim(col("product_name"))))


In [0]:
#validate price
valid_price = standard_name.filter(
                    (col('price') > 0)
                    & (col('price').isNotNull()))


df_quarantine = standard_name.filter(
                    (~((col('price') > 0)
                    & (col('price').isNotNull())
                    ))).withColumn("validation_status", lit("invalid_price"))

#quarantine rejected rows from validation
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.products")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.product_id = source.product_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()


In [0]:
#validate cost
valid_cost = valid_price.filter(
                    (col('cost') > 0)
                    & (col('cost').isNotNull()))


df_quarantine = valid_price.filter(
                    (~((col('cost') > 0)
                    & (col('cost').isNotNull())
                    ))).withColumn("validation_status", lit("invalid_cost"))


#quarantine rejected rows from validation
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.products")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.product_id = source.product_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#Validate category

expected_category = {"Electronics","Home","Fashion","Sports","Beauty"}

df_valid = valid_cost.filter(col("category").isin(expected_category))

df_quarantine = valid_cost.filter(~col("category").isin(expected_category)
                                      ).withColumn("validation_status", lit("invalid_category"))

#quarantine rejected rows from validation
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.products")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.product_id = source.product_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#Standardize product status
df_valid = df_valid.withColumn("product_status", initcap(trim(col("product_status"))))